# 🎯 Feature Engineering

## Objective

Transform the cleaned dataset into meaningful and model-ready features while preserving useful information for predicting student placement.

In [61]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/student_placement_clean.csv")
df.shape

(100000, 17)

In [3]:
technical_skill_score = df[["coding_skills", "dsa_score", "ml_knowledge", "system_design"]].mean(axis=1)
technical_skill_score

0        4.675
1        5.375
2        5.500
3        4.025
4        4.025
         ...  
99995    5.350
99996    4.900
99997    5.675
99998    4.850
99999    4.300
Length: 100000, dtype: float64

In [4]:
technical_skill_score.describe()

count    100000.000000
mean          5.003205
std           0.882331
min           1.350000
25%           4.400000
50%           5.000000
75%           5.600000
max           8.475000
dtype: float64

In [6]:
df["technical_skill_score"] = technical_skill_score
df.shape

(100000, 18)

## Feature Engineering: Technical Skill Score

### Observation:

- Created a new `technical_skill_score` feature by taking the average of `coding_skills`, `dsa_score`, `ml_knowledge`, and `system_design`.
- The new feature contains **100,000 observations** with no missing values.
- The score has a mean of approximately **5.00** and a median of **5.00**.
- The feature combines multiple technical skill indicators into a single overall technical skill measure.

In [12]:
df["has_backlog"] = (df["backlogs"] > 0).astype(int)
df[["backlogs", "has_backlog"]].head(10)

,backlogs,has_backlog
0,0,0
1,0,0
2,0,0
3,0,0
4,1,1
5,1,1
6,0,0
7,0,0
8,0,0
9,3,1


## Feature Engineering: Backlog Indicator

### Observation:

- Created a new binary `has_backlog` feature from the `backlogs` column.
- Students with no backlogs are assigned **0**.
- Students with one or more backlogs are assigned **1**.
- The original `backlogs` feature is retained because it contains additional information about the number of backlogs.

In [14]:
experience_score = df[[
    "internships",
    "projects_count",
    "certifications",
    "hackathons",
    "open_source_contributions",
    "extracurriculars"
]].sum(axis=1)

In [15]:
experience_score.describe()

count    100000.000000
mean          7.340100
std           2.415775
min           0.000000
25%           6.000000
50%           7.000000
75%           9.000000
max          17.000000
dtype: float64

In [17]:
df["experience_score"] = experience_score
df.shape

(100000, 20)

## Feature Engineering: Experience Score

### Observation:

- Created a new `experience_score` feature by summing internships, projects, certifications, hackathons, open-source contributions, and extracurricular activities.
- The feature represents the overall level of student experience and participation in different activities.
- The score ranges from **0 to 17**, with an average of approximately **7.34**.
- No missing values were present in the newly created feature.

In [18]:
maximum_skill = df[["coding_skills",
"dsa_score",
"ml_knowledge",
"system_design"]].max(axis=1)

In [19]:
minimum_skill = df[["coding_skills",
"dsa_score",
"ml_knowledge",
"system_design"]].min(axis=1)

In [21]:
technical_skill_gap = maximum_skill - minimum_skill
technical_skill_gap

0        7.3
1        6.0
2        2.4
3        5.6
4        3.2
        ... 
99995    6.0
99996    4.4
99997    5.6
99998    7.8
99999    2.9
Length: 100000, dtype: float64

In [22]:
technical_skill_gap.describe()

count    100000.000000
mean          4.105301
std           1.672523
min           0.100000
25%           2.900000
50%           4.000000
75%           5.200000
max          10.000000
dtype: float64

In [23]:
df["technical_skill_gap"] = technical_skill_gap
df.shape

(100000, 21)

## Feature Engineering: Technical Skill Gap

### Observation:

- Created `technical_skill_gap` by subtracting the minimum technical skill from the maximum technical skill for each student.
- The feature measures how balanced or uneven a student's technical skills are.
- The score ranges from **0.1 to 10.0**, with an average of approximately **4.11**.
- A lower gap indicates more balanced technical skills, while a higher gap indicates greater variation between technical strengths and weaknesses.
- The feature contains **100,000 observations** with no missing values.

In [25]:
df[[
    "technical_skill_score",
    "has_backlog",
    "experience_score",
    "technical_skill_gap",
    "placement_status"
]].corr()["placement_status"]

technical_skill_score    0.081257
has_backlog             -0.049564
experience_score         0.117112
technical_skill_gap      0.043018
placement_status         1.000000
Name: placement_status, dtype: float64

## Validation of Engineered Features

### Observation:

- `experience_score` has the strongest positive correlation with placement status among the engineered features, at approximately **0.12**.
- `technical_skill_score` has a positive correlation of approximately **0.08** with placement status.
- `technical_skill_gap` shows a weak positive correlation of approximately **0.04**.
- `has_backlog` has a weak negative correlation of approximately **-0.05**.
- The correlations are relatively weak, indicating that the engineered features should be considered together with the original features rather than as individual predictors.
- The usefulness of these features will ultimately be evaluated during model training and evaluation.

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   branch                     100000 non-null  str    
 1   college_tier               100000 non-null  str    
 2   cgpa                       100000 non-null  float64
 3   backlogs                   100000 non-null  int64  
 4   coding_skills              100000 non-null  float64
 5   dsa_score                  100000 non-null  float64
 6   aptitude_score             100000 non-null  float64
 7   communication_skills       100000 non-null  float64
 8   ml_knowledge               100000 non-null  float64
 9   system_design              100000 non-null  float64
 10  internships                100000 non-null  int64  
 11  projects_count             100000 non-null  int64  
 12  certifications             100000 non-null  int64  
 13  hackathons                 100000 non-nul

## Feature Engineering Validation

### Observation:

- The engineered dataset contains **100,000 rows and 21 columns**.
- All features contain **100,000 non-null values**, confirming that no missing values were introduced during feature engineering.
- The four newly created features have appropriate numerical data types.
- The categorical features `branch` and `college_tier` remain unchanged.
- The target variable `placement_status` remains an integer feature.
- The dataset is ready for the next stage of model preparation.

In [30]:
X = df.drop(columns=["placement_status"])
y = df["placement_status"]

In [32]:
X.shape

(100000, 20)

In [33]:
y.shape

(100000,)

## Feature and Target Separation

### Observation:

- The target variable `placement_status` was separated from the input features.
- `X` contains **20 features** that will be used for prediction.
- `y` contains the **placement_status** target variable.
- Both `X` and `y` contain **100,000 student records**.
- The dataset is now prepared for the next stage of model preprocessing.

In [34]:
categorical_features = ["branch", "college_tier"]
categorical_features

['branch', 'college_tier']

In [35]:
numerical_features = X.drop(columns=categorical_features).columns.tolist()
numerical_features

['cgpa',
 'backlogs',
 'coding_skills',
 'dsa_score',
 'aptitude_score',
 'communication_skills',
 'ml_knowledge',
 'system_design',
 'internships',
 'projects_count',
 'certifications',
 'hackathons',
 'open_source_contributions',
 'extracurriculars',
 'technical_skill_score',
 'has_backlog',
 'experience_score',
 'technical_skill_gap']

In [36]:
len(numerical_features)

18

## Identifying Feature Types

### Observation:

- The dataset contains **20 input features** after separating the target variable.
- **2 features** are categorical: `branch` and `college_tier`.
- **18 features** are numerical.
- The engineered features are included among the numerical features.
- The categorical features will be one-hot encoded, while numerical features will remain numerical for the next preprocessing step.

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", "passthrough", numerical_features)
    ]
)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [49]:
X_train.shape

(80000, 20)

In [50]:
X_test.shape

(20000, 20)

In [51]:
y_train.shape

(80000,)

In [54]:
y_test.shape

(20000,)

## Train-Test Split

### Observation:

- The dataset was divided into **80% training data** and **20% testing data**.
- The training set contains **80,000 students**, while the testing set contains **20,000 students**.
- `stratify=y` was used to preserve the placement-status class distribution in both sets.
- `random_state=42` was used to make the split reproducible.
- The training and testing data are now ready for preprocessing.

In [55]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("X_train encoded shape:", X_train_encoded.shape)
print("X_test encoded shape:", X_test_encoded.shape)

X_train encoded shape: (80000, 28)
X_test encoded shape: (20000, 28)


In [56]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 28
['categorical__branch_CE' 'categorical__branch_CSE'
 'categorical__branch_Chemical' 'categorical__branch_ECE'
 'categorical__branch_EE' 'categorical__branch_IT'
 'categorical__branch_ME' 'categorical__college_tier_Tier-1'
 'categorical__college_tier_Tier-2' 'categorical__college_tier_Tier-3'
 'numerical__cgpa' 'numerical__backlogs' 'numerical__coding_skills'
 'numerical__dsa_score' 'numerical__aptitude_score'
 'numerical__communication_skills' 'numerical__ml_knowledge'
 'numerical__system_design' 'numerical__internships'
 'numerical__projects_count' 'numerical__certifications'
 'numerical__hackathons' 'numerical__open_source_contributions'
 'numerical__extracurriculars' 'numerical__technical_skill_score'
 'numerical__has_backlog' 'numerical__experience_score'
 'numerical__technical_skill_gap']


## Categorical Encoding

### Observation:

- The categorical features `branch` and `college_tier` were transformed using **One-Hot Encoding**.
- The 18 numerical features were kept unchanged using `passthrough`.
- The original 20 features became **28 model-ready features** after encoding.
- `branch` produced 7 encoded features and `college_tier` produced 3 encoded features.
- The four engineered features were successfully included in the transformed dataset.
- The same fitted preprocessor was used to transform both training and testing data, preventing data leakage.

In [58]:
print("Training data:")
print("Shape:", X_train_encoded.shape)
print("Contains NaN:", np.isnan(X_train_encoded).any())

print("\nTesting data:")
print("Shape:", X_test_encoded.shape)
print("Contains NaN:", np.isnan(X_test_encoded).any())

Training data:
Shape: (80000, 28)
Contains NaN: False

Testing data:
Shape: (20000, 28)
Contains NaN: False


## Validation of Transformed Data

### Observation:

- The encoded training data contains **80,000 samples and 28 features**.
- The encoded testing data contains **20,000 samples and 28 features**.
- No missing (`NaN`) values were introduced during preprocessing.
- Both training and testing data have the same number of features.
- The transformed data is ready for the final preparation step.

In [60]:
joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']

In [62]:
os.path.exists("../models/preprocessor.pkl")

True

## Saving the Preprocessing Pipeline

### Observation:

- The fitted preprocessing pipeline was saved as `preprocessor.pkl`.
- The pipeline contains the One-Hot Encoding configuration learned from the training data.
- Saving the fitted preprocessor allows the same transformations to be applied consistently to future data.
- The preprocessing pipeline is now ready to be reused during model training and prediction.